# Module 3 — Topic 6: Data Storytelling
## Live Demo Notebook

**Scenario:** You've completed your Jumia Nigeria Q3 sales analysis.
Now your manager asks: *"What's the most important finding? Give me one chart and tell me what to do about it."*

This demo shows the full journey:
1. The messy exploratory chart (what you made during EDA)
2. The polished explanatory chart (what you show your manager)
3. The complete three-act data story

---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style='whitegrid')

df = pd.read_csv('jumia_sales.csv')
print('Dataset:', df.shape)

# Compute the key dataset we'll use throughout
revenue_by_cat = df.groupby('category')['revenue'].sum().sort_values(ascending=False)

---
## Part 1 — The Exploratory Chart

This is what you make **during** EDA — for yourself, not for your audience.

In [ ]:
# EXPLORATORY — all categories, default colours, generic title
fig, ax = plt.subplots(figsize=(12, 5))

revenue_by_cat.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')

ax.set_title('Revenue by Category')   # generic — says nothing
ax.set_xlabel('Product Category')
ax.set_ylabel('Total Revenue (₦)')
ax.tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()

print('This chart is fine for exploration.')
print('But it has no focus, no emphasis, no finding — it just shows the data.')

---
## Part 2 — Finding the Story

Before building the explanatory chart, we need to commit to ONE finding.

In [ ]:
# What is the most compelling finding in this dataset?
top_cat    = revenue_by_cat.index[0]
bottom_cat = revenue_by_cat.index[-1]
top_rev    = revenue_by_cat.iloc[0]
bottom_rev = revenue_by_cat.iloc[-1]
ratio      = top_rev / bottom_rev

print('=== Key Finding ===')
print(f'Highest revenue: {top_cat:<25} ₦{top_rev:,.0f}')
print(f'Lowest revenue:  {bottom_cat:<25} ₦{bottom_rev:,.0f}')
print(f'Ratio: {ratio:.1f}×')
print()
print('>>> STORY: The top category generates {:.0f}× more revenue than the bottom.'.format(ratio))
print(f'>>> This is a significant gap worth highlighting.')

---
## Part 3 — Building the Explanatory Chart

Step by step: from generic to polished.

In [ ]:
# Step 1: Highlight the two key categories — grey out the rest
highlight = [top_cat, bottom_cat]
highlight_colour = '#E63946'   # red for emphasis
context_colour   = '#CED4DA'   # light grey for context

colours = [highlight_colour if c in highlight else context_colour
           for c in revenue_by_cat.index]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(revenue_by_cat.index, revenue_by_cat.values,
              color=colours, edgecolor='white')

ax.set_title('Revenue by Category — Step 1: Highlight the key bars')
ax.set_ylabel('Total Revenue (₦)')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()

In [ ]:
# Step 2: Add value labels to the highlighted bars
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(revenue_by_cat.index, revenue_by_cat.values,
              color=colours, edgecolor='white')

for bar, label in zip(bars, revenue_by_cat.index):
    if label in highlight:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, height + revenue_by_cat.max() * 0.02,
                f'₦{height/1e6:.1f}M', ha='center', fontsize=11,
                fontweight='bold', color=highlight_colour)

ax.set_title('Revenue by Category — Step 2: Add value labels')
ax.set_ylabel('Total Revenue (₦)')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()

In [ ]:
# Step 3: Change the title to state the finding
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(revenue_by_cat.index, revenue_by_cat.values,
              color=colours, edgecolor='white')

for bar, label in zip(bars, revenue_by_cat.index):
    if label in highlight:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, height + revenue_by_cat.max() * 0.02,
                f'₦{height/1e6:.1f}M', ha='center', fontsize=11,
                fontweight='bold', color=highlight_colour)

# FINDING-DRIVEN TITLE — states the conclusion, not just the data
ax.set_title(f'{top_cat} Generates {ratio:.0f}× More Revenue Than {bottom_cat}',
             fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Total Revenue (₦)', fontsize=12)
ax.tick_params(axis='x', rotation=35)
ax.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Step 4: Final polish — add annotation arrow, clean up, add legend
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(revenue_by_cat.index, revenue_by_cat.values,
              color=colours, edgecolor='white')

# Value labels on highlighted bars
for bar, label in zip(bars, revenue_by_cat.index):
    if label in highlight:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, height + revenue_by_cat.max() * 0.02,
                f'₦{height/1e6:.1f}M', ha='center', fontsize=12,
                fontweight='bold', color=highlight_colour)

# Annotation arrow pointing to the bottom bar
bottom_bar_x = list(revenue_by_cat.index).index(bottom_cat)
ax.annotate(f'{ratio:.0f}× gap',
            xy=(bottom_bar_x, revenue_by_cat[bottom_cat]),
            xytext=(bottom_bar_x + 2.5, revenue_by_cat.max() * 0.55),
            arrowprops=dict(arrowstyle='->', color='#333333', lw=1.8),
            fontsize=12, fontweight='bold', color='#333333')

# Title as finding
ax.set_title(f'{top_cat} Generates {ratio:.0f}× More Revenue Than {bottom_cat}',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Product Category', fontsize=11)
ax.set_ylabel('Total Revenue (₦)', fontsize=11)
ax.tick_params(axis='x', rotation=35)
ax.grid(axis='y', linestyle='--', alpha=0.25)

# Simple legend
legend_patches = [
    mpatches.Patch(color=highlight_colour, label='Key categories'),
    mpatches.Patch(color=context_colour,   label='Other categories'),
]
ax.legend(handles=legend_patches, loc='upper right', fontsize=10)

# Source note
fig.text(0.99, 0.01, 'Source: Jumia Nigeria Q3 Orders Export | 500 orders, 8 categories',
         ha='right', fontsize=8, color='grey')

plt.tight_layout()
plt.show()

print('This is the explanatory chart — presentation-ready.')

---
## Part 4 — The Complete Three-Act Data Story

Context → Finding → Recommendation

### Act 1 — Context

> We analysed **500 Jumia Nigeria Q3 orders** across **8 product categories** and **15 states**.
> The dataset covers unit prices from ₦800 to ₦320,000 and total revenues per order from ₦800 to over ₦1.9M.

### Act 2 — Finding

> *(See chart above)*

### Act 3 — Recommendation

> **Recommendation:** Increase stock depth and marketing investment in **Phones & Tablets** — the highest-revenue category by a significant margin.
> **Investigate** the **Books & Stationery** category: despite processing a comparable number of orders, it generates disproportionately low revenue. Review pricing strategy and margin viability.

---
## Part 5 — Bonus: Supporting Context Chart

One explanatory chart isn't always enough. Add a second chart for supporting context — but keep it focused.

In [ ]:
# Supporting context: order COUNT vs REVENUE — is the count gap as large as the revenue gap?
count_by_cat = df.groupby('category').size().reindex(revenue_by_cat.index)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: revenue
ax1.bar(revenue_by_cat.index, revenue_by_cat.values,
        color=colours, edgecolor='white')
ax1.set_title('Total Revenue by Category', fontsize=12, fontweight='bold')
ax1.set_ylabel('Total Revenue (₦)')
ax1.tick_params(axis='x', rotation=40)

# Right: order count
ax2.bar(count_by_cat.index, count_by_cat.values,
        color=colours, edgecolor='white')
ax2.set_title('Number of Orders by Category', fontsize=12, fontweight='bold')
ax2.set_ylabel('Number of Orders')
ax2.tick_params(axis='x', rotation=40)

fig.suptitle('Revenue Gap Is Not Explained by Order Count',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Calculate the revenue-per-order ratio
rev_per_order = (revenue_by_cat / count_by_cat).sort_values(ascending=False)
print('Average revenue per order by category:')
for cat, val in rev_per_order.items():
    print(f'  {cat:<25} ₦{val:,.0f}')

### Story Addition:

> The revenue gap is **not simply a volume effect** — Phones & Tablets processes a similar number of orders as Books & Stationery, but each order is worth dramatically more.
> This confirms that the category difference is driven by **product value**, not order frequency.
> The recommendation stands: prioritise Phones & Tablets for stock and marketing.

---
## Part 6 — Checklist: Is Your Chart Story-Ready?

In [ ]:
checklist = [
    ('Title states the finding (not just the data)',      True),
    ('Axis labels include units',                         True),
    ('Key insight is annotated',                          True),
    ('Non-essential elements removed',                    True),
    ('Chart answers exactly one question',                True),
]

print('=== Story Chart Checklist ===')
for item, passed in checklist:
    status = 'PASS' if passed else 'FAIL'
    print(f'  [{status}] {item}')

score = sum(p for _, p in checklist)
print(f'\nScore: {score}/{len(checklist)}')
if score == len(checklist):
    print('>>> Chart is presentation-ready.')
else:
    print('>>> Revisit the failing items before sharing.')

---
## Summary

In this demo we:
- Started with a **messy exploratory chart** — all categories, no focus
- Identified the **key finding**: the top/bottom revenue gap
- Built the **explanatory chart** in four steps: colour → labels → title → annotation
- Framed the complete **three-act data story**: context → finding → recommendation
- Added a **supporting context chart** to strengthen the recommendation
- Applied the **five-point story chart checklist**

---

## Module 3 Complete — EDA & Visualization

You can now:
- Run a structured EDA on any dataset (Topic 1)
- Detect outliers and interpret skewness (Topic 2)
- Build charts with Matplotlib (Topic 3) and Seaborn (Topic 4)
- Choose the right chart for any analytical question (Topic 5)
- Turn a finding into a clear, annotated, actionable data story (Topic 6)

**Next — Module Demo:** A combined notebook applying all six topics to one continuous health insurance dataset.